In [0]:
# --------------------------------------------------------------
# Imports
# --------------------------------------------------------------
from pyspark.sql import SparkSession, functions as F, types as T, Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
# Isolation Forest from scikit-learn (spark-iforest isn't available in this workspace)
from sklearn.ensemble import IsolationForest
import mlflow
import mlflow.sklearn
import numpy as np
import uuid

# --------------------------------------------------------------
# Spark session (Databricks already provides one)
# --------------------------------------------------------------
spark = SparkSession.builder.getOrCreate()

# --------------------------------------------------------------
# MLflow tracking URI – adjust if you use a custom server
# --------------------------------------------------------------
# By default Databricks notebooks use the workspace‑level tracking server.
# If you run elsewhere, point to your server, e.g.:
# mlflow.set_tracking_uri("http://my-mlflow-server:5000")
mlflow.set_experiment("/Shared/employee_risk_detection")   # change path as you wish

<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/2218560781371264', creation_time=1794083422638, experiment_id='2218560781371264', last_update_time=1794084174925, lifecycle_stage='active', name='/Shared/employee_risk_detection', tags={'mlflow.experiment.sourceName': '/Shared/employee_risk_detection',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'hemamrutha.k7@gmail.com',
 'mlflow.ownerId': '72410163837737'}, trace_location=None, workspace='default'>

In [0]:
# DBFS mounts (/mnt/...) are disabled in this workspace — load from Unity Catalog instead
gold_df = spark.read.table("workspace.default.gold_features")

# sanity check
display(gold_df.select("employee_id","date","files_cnt","files_z","usb_z","after_hours_z").limit(5))

employee_id,date,files_cnt,files_z,usb_z,after_hours_z
1,2023-10-01,9,0.13988620785159123,-0.2776734596426306,-1.067075586239545
1,2023-10-02,5,-1.5387482863675004,0.5156792821934576,-1.067075586239545
1,2023-10-03,10,0.5595448314063641,0.5156792821934576,0.9336911379596
1,2023-10-04,7,-0.6994310392579546,2.1023847658656343,-0.40015334483982995
1,2023-10-05,9,0.13988620785159123,1.3090320240295459,0.26676889655988506


In [0]:
feature_cols = [
    "files_cnt",
    "confidential_files_cnt",
    "usb_cnt",
    "external_emails_cnt",
    "after_hours_cnt",
    "files_z",
    "confidential_files_z",
    "usb_z",
    "external_emails_z",
    "after_hours_z"
]

# VectorAssembler → raw feature vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")

# Optional standard‑scaler (IsolationForest works fine without it, but scaling helps the distance‑based tree splits)
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)

# --------------------------------------------------------------
# Build a pipeline that only does the preprocessing (no model yet)
# --------------------------------------------------------------
prep_pipe = Pipeline(stages=[assembler, scaler])
prep_model = prep_pipe.fit(gold_df)
prepared_df = prep_model.transform(gold_df)

prepared_df.select("employee_id","date","features").printSchema()

root
 |-- employee_id: long (nullable = true)
 |-- date: date (nullable = true)
 |-- features: vectorudt (nullable = true)



In [0]:
# --------------------------------------------------------------
# Isolation Forest hyper‑parameters (feel free to tune)
# --------------------------------------------------------------
# scikit‑learn IsolationForest — spark-iforest isn't available in this workspace.
# numTrees → n_estimators; maxDepth has no direct sklearn equivalent
# (sklearn controls tree depth via sub‑sampling), so it is dropped.
iso = IsolationForest(
    n_estimators=200,         # was numTrees
    contamination=0.02,       # we expect ~2 % malicious days
    random_state=42           # was seed
)

# --------------------------------------------------------------
# Fit the model
# --------------------------------------------------------------
# The dataset is tiny (one row per employee‑day, ≈ 150 × 30 rows), so we
# collect the assembled feature vectors and fit on the driver.
with mlflow.start_run(run_name="IsolationForest_employee_risk") as run:
    # Log parameters
    mlflow.log_param("contamination", 0.02)
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("library", "scikit-learn")

    # Collect the assembled feature vectors (small data → safe on the driver)
    feature_rows = prepared_df.select("employee_id", "date", "features").collect()
    X = np.array([row["features"].toArray() for row in feature_rows])

    # Fit
    iso.fit(X)

    # Score every row: −decision_function → higher = more anomalous,
    # then min‑max rescale to 0‑1 (matches the old anomaly_score semantics)
    raw_scores = -iso.decision_function(X)
    s_min, s_max = float(raw_scores.min()), float(raw_scores.max())
    if s_max > s_min:
        norm_scores = (raw_scores - s_min) / (s_max - s_min)
    else:
        norm_scores = np.zeros_like(raw_scores)

    # sklearn predict: −1 = anomaly, 1 = normal → recode to 1 = anomaly, 0 = normal
    anomaly_flags = (iso.predict(X) == -1).astype(int)

    # Attach the scores back to the Spark DataFrame
    score_rows = [
        (r["employee_id"], r["date"], float(s), int(p))
        for r, s, p in zip(feature_rows, norm_scores, anomaly_flags)
    ]
    scores_df = spark.createDataFrame(
        score_rows, ["employee_id", "date", "anomaly_score", "prediction"]
    )
    scored_df = prepared_df.join(scores_df, on=["employee_id", "date"], how="left")

    # Log the scikit‑learn model
    mlflow.sklearn.log_model(iso, artifact_path="isolation_forest_model")

    # Compute a quick “baseline” metric: average anomaly score on the whole set
    avg_score = scored_df.agg(F.mean("anomaly_score")).first()[0]
    mlflow.log_metric("avg_anomaly_score", avg_score)

    # (Optional) Log a few sample rows as an artifact for inspection
    # (/tmp is not writable on this compute → log as an MLflow artifact instead)
    sample = (
        scored_df.select("employee_id", "date", "anomaly_score", "prediction")
        .orderBy(F.desc("anomaly_score"))
        .limit(20)
        .toPandas()
    )
    mlflow.log_dict(
        {"rows": sample.to_dict(orient="records")},
        "sample_predictions/sample_predictions.json",
    )

2026/11/07 20:45:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-0e69f6f2-603b.cloud.databricks.com/ml/experiments/2218560781371264/models/m-0aecc8ecdba84e8d999125446918c1c6?o=7474658800894907
2026/11/07 20:45:31 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.12.0/ml/model/signatures.html for instructions on setting signature on models.


In [0]:
rule_df = (
    scored_df
    .withColumn("high_file_z",          F.when(F.col("files_z") > 2, 1).otherwise(0))
    .withColumn("high_confidential_z",  F.when(F.col("confidential_files_z") > 2, 1).otherwise(0))
    .withColumn("high_usb_z",           F.when(F.col("usb_z") > 2, 1).otherwise(0))
    .withColumn("high_email_z",         F.when(F.col("external_emails_z") > 2, 1).otherwise(0))
    .withColumn("high_afterhours_z",    F.when(F.col("after_hours_z") > 2, 1).otherwise(0))
    .withColumn("any_after_hours_event",F.when(F.col("after_hours_cnt") > 0, 1).otherwise(0))
)

In [0]:
# ----------  Weights (tweak as needed) ----------
w_model      = 0.6   # 60 % of total risk comes from the ML model
w_rule_total = 0.4   # remaining 40 % is from rule‑based alerts

# each rule gets an equal slice of the rule budget
num_rules = 6
rule_weight = w_rule_total / num_rules

risk_df = (
    rule_df
    .withColumn(
        "rule_score",
        (F.col("high_file_z")          * rule_weight) +
        (F.col("high_confidential_z")  * rule_weight) +
        (F.col("high_usb_z")           * rule_weight) +
        (F.col("high_email_z")         * rule_weight) +
        (F.col("high_afterhours_z")    * rule_weight) +
        (F.col("any_after_hours_event")* rule_weight)
    )
    .withColumn(
        "risk_score",
        F.round(F.col("anomaly_score") * w_model + F.col("rule_score"), 4)
    )
)

# Keep a tidy column set for the gold table
gold_risk = risk_df.select(
    "employee_id",
    "date",
    "files_cnt",
    "confidential_files_cnt",
    "usb_cnt",
    "external_emails_cnt",
    "after_hours_cnt",
    "files_z",
    "confidential_files_z",
    "usb_z",
    "external_emails_z",
    "after_hours_z",
    "anomaly_score",
    "prediction",
    "high_file_z",
    "high_confidential_z",
    "high_usb_z",
    "high_email_z",
    "high_afterhours_z",
    "any_after_hours_event",
    "risk_score"
)

display(gold_risk.orderBy(F.desc("risk_score")).limit(10))

employee_id,date,files_cnt,confidential_files_cnt,usb_cnt,external_emails_cnt,after_hours_cnt,files_z,confidential_files_z,usb_z,external_emails_z,after_hours_z,anomaly_score,prediction,high_file_z,high_confidential_z,high_usb_z,high_email_z,high_afterhours_z,any_after_hours_event,risk_score
137,2023-10-07,19,4,5,6,26,3.326965928601664,2.8709162350554047,-1.225176043688485,-0.0545167200456898,2.5519548706885837,1.0,1,1,1,0,0,1,1,0.8667
124,2023-10-17,17,4,12,12,25,2.0408029185890895,0.7045015893465368,0.3162407328140147,2.4396829842835945,0.9358534883192888,0.9460433109159503,1,1,0,0,1,0,1,0.7676
79,2023-10-20,18,4,7,2,19,3.0842468475062432,2.7144021446028717,-0.5232549675626575,-2.391766699577005,-0.7988224702933557,0.9456047818017755,1,1,1,0,0,0,1,0.7674
25,2023-10-06,15,5,4,7,24,2.1730708456108907,3.496435707494362,-1.9266780557420127,0.12984785372226643,1.0521121913044644,0.8800360712261763,1,1,1,0,0,0,1,0.728
131,2023-10-21,17,5,5,6,23,3.201965218729005,3.088084500892581,-1.2870921558367838,-0.11140395708761369,1.0864025842070761,0.878705859492551,1,1,1,0,0,0,1,0.7272
59,2023-10-19,17,2,9,4,27,2.9602322589865153,1.4169305938161922,-0.3549666995272258,-0.8971458019688348,2.8226868703619337,0.8422864183199067,1,1,0,0,0,1,1,0.7054
107,2023-10-16,18,3,6,2,21,2.407692646363978,2.400028636713462,-0.9902824996085516,-1.9663841605003503,-0.16861605068622446,0.840925930702916,1,1,1,0,0,0,1,0.7046
24,2023-10-16,17,6,4,8,19,2.400322210812558,1.6595603747721916,-1.98139540283038,0.6204480253916433,-1.3801726697320698,0.9477517200923332,1,1,0,0,0,0,1,0.702
11,2023-10-10,5,0,13,13,25,-1.1659725709295816,-0.7897188828807975,1.4073544719753277,2.6597390159179297,2.147225208566318,0.8265847221353498,1,0,0,0,1,1,1,0.696
32,2023-10-16,17,2,3,4,15,3.009533107928992,0.9128709291752767,-2.3709629596021196,-1.030225801046036,-2.6713187430557497,0.9360271017076375,1,1,0,0,0,0,1,0.6949


In [0]:
# DBFS mounts (/mnt/...) are disabled in this workspace —
# persist as a Unity Catalog managed table instead
gold_risk.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_risk")

print("✅ Gold risk table 'workspace.default.gold_risk' created and registered.")

✅ Gold risk table 'workspace.default.gold_risk' created and registered.
